In [9]:
import numpy as np
import pandas as pd
# ruff: noqa: E501
#
# cdf_manager = ImapCdfAttributes()
# cdf_manager.add_instrument_global_attrs("ultra")
# cdf_manager.add_instrument_variable_attrs("ultra", "l1b")
#
# folder_path = (
#     "/Users/luco3133/projects/ultra_stuff/validation_stuff"
#     "/other_var_validation_20251024/ultra-45-inputs"
# )
#
# kernles = [
#     "/Users/luco3133/projects/imap_processing/imap_processing/"
#     "tests/spice/test_data/imap_sclk_0000.tsc",
#     "/Users/luco3133/projects/imap_processing/imap_processing/"
#     "tests/spice/test_data/naif0012.tls",
# ]
#
#
# def vectorijk_to_theta_phi(x_inst, y_inst, z_inst):
#     """Get theta and phi."""
#     x_permuted = z_inst  # K
#     y_permuted = y_inst  # J
#     z_permuted = x_inst  # I
#
#     # Declination (theta)
#     theta_rad = np.arcsin(np.clip(z_permuted, -1.0, 1.0))
#
#     # Right Ascension (phi)
#     phi_rad = np.arctan2(y_permuted, x_permuted)
#
#     # Convert to degrees
#     theta_deg = np.degrees(theta_rad)
#     phi_deg = np.degrees(phi_rad)
#
#     return theta_deg, phi_deg
#
#
# def get_scattering_threshold_for_energy(energy):
#     """Get scattering threshold based on energy."""
#     if energy < 5:
#         return 12.0
#     elif energy < 8:
#         return 10.0
#     elif energy < 10:
#         return 8.0
#     elif energy < 20:
#         return 6.0
#     elif energy < 400:
#         return 4.0
#
#
# files = os.listdir(folder_path)
# pointings = [f.split("-")[-1].replace(".csv", "") for f in files]
# pointings = ["p0"]
#
# # You'll need to provide ancillary_files - adjust path as needed
# ancillary_files = {
#     "l1b-90sensor-scattering-calibration-data": "/Users/luco3133/projects/imap_processing/data/imap/ancillary/ultra/imap_ultra_l1b-90sensor-scattering-calibration-data_20250101_v000.csv",
#     "l1b-scattering-thresholds-per-energy": "/Users/luco3133/projects/imap_processing/data/imap/ancillary/ultra/imap_ultra_l1b-scattering-thresholds-per-energy_20250101_v000.csv",
# }
#
# with sp.KernelPool(kernles) as pool:
#     for pointing in pointings:
#         for id in [45]:
#             print(
#                 f"Creating direct event dataset for {pointing} pointing and {id} sensor"
#             )
#
#             ae_flux = pd.read_csv(
#                 os.path.join(folder_path, f"AE-IMAP_ULTRA_{id}-{pointing}.csv")
#             )
#             rates_flux = pd.read_csv(
#                 f"{folder_path}/Rates-IMAP_ULTRA_{id}-{pointing}.csv"
#             )
#
#             # energy_max = 1000
#
#             # Get event energies
#             event_energies = ae_flux["energy_sc"].values
#
#             print(f"  Processing {len(event_energies)} events")
#
#             # Calculate theta/phi from instrument direction vectors using Java's method
#             print(
#                 "  Calculating theta/phi from instrument direction vectors (Java method)..."
#             )
#             x_inst = ae_flux["x_inst"].values
#             y_inst = ae_flux["y_inst"].values
#             z_inst = ae_flux["z_inst"].values
#
#             theta, phi = vectorijk_to_theta_phi(x_inst, y_inst, z_inst)
#
#             print(f"  Theta range: [{np.min(theta):.2f}, {np.max(theta):.2f}] deg")
#             print(f"  Phi range: [{np.min(phi):.2f}, {np.max(phi):.2f}] deg")
#
#             # Apply scattering filter using energy bin geometric means
#             print("\n  === APPLYING SCATTERING FILTER ===")
#
#             # Get energy bins
#             intervals, _, energy_bin_geometric_means = build_energy_bins()
#             print(intervals)
#             intervals = np.array(intervals)
#             energy_bin_geometric_means = np.array(energy_bin_geometric_means)
#
#             # FILTER: Only process events within the valid energy range
#             energy_in_range = (event_energies >= intervals[0, 0]) & (
#                 event_energies <= intervals[-1, 1]
#             )
#             print(f"Energy range: {intervals[0, 0]:.1f} - {intervals[-1, 1]:.1f} keV")
#             print(f"Events in range: {np.sum(energy_in_range)}/{len(event_energies)}")
#
#             # Initialize quality flags for ALL events (keep original size)
#             # Flag events outside energy range as outliers
#             # Initialize quality flags for ALL events (keep original size)
#             scattering_quality_flags = np.zeros(
#                 len(event_energies), dtype=np.uint16
#             )  # ← ADD THIS
#
#             # Flag events outside energy range as outliers
#             outlier_quality_flags = np.zeros(len(event_energies), dtype=np.uint16)
#             outlier_quality_flags[~energy_in_range] = 1
#             # np.testing.assert_array_equal(outlier_quality_flags[:9], ~[False, False, True, False, False, True, False, True, True])
#             print(f"Outlier quality flags: {outlier_quality_flags}")
#             # Determine which energy bin each event falls into (for all events)
#             bin_indices = np.digitize(event_energies, intervals[:, 0]) - 1
#             bin_indices = np.clip(bin_indices, 0, len(energy_bin_geometric_means) - 1)
#
#             print(f"Bin indices: {bin_indices}")
#             # Process each energy bin
#             for ebin in range(len(energy_bin_geometric_means)):
#                 # Get events in THIS bin only
#                 event_mask = (bin_indices == ebin) & energy_in_range
#
#                 if not np.any(event_mask):
#                     continue
#
#                 # Get energy bin geometric mean
#                 energy_geom_mean = energy_bin_geometric_means[ebin]
#
#                 # Get threshold for this energy
#                 threshold = get_scattering_threshold_for_energy(energy_geom_mean)
#
#                 # Get scattering coefficients for events in this bin
#                 theta_coeffs, phi_coeffs = get_scattering_coefficients(
#                     theta[event_mask],
#                     phi[event_mask],
#                     lookup_tables=None,
#                     ancillary_files=ancillary_files,
#                     instrument_id=id,
#                 )
#                 # After getting coefficients
#                 has_nan_coeff = (
#                     np.isnan(theta_coeffs[:, 0])
#                     | np.isnan(theta_coeffs[:, 1])
#                     | np.isnan(phi_coeffs[:, 0])
#                     | np.isnan(phi_coeffs[:, 1])
#                 )
#
#                 # Flag events with NaN coefficients
#                 event_indices = np.where(event_mask)[0]
#                 scattering_quality_flags[event_indices[has_nan_coeff]] |= (
#                     ImapDEScatteringUltraFlags.NAN_PHI_OR_THETA.value
#                 )
#
#                 # For valid coefficients, calculate FWHM and check threshold
#                 valid_coeffs = ~has_nan_coeff
#                 if np.any(valid_coeffs):
#                     fwhm_theta_valid = (
#                         theta_coeffs[valid_coeffs, 0]
#                         * energy_geom_mean ** theta_coeffs[valid_coeffs, 1]
#                     )
#                     fwhm_phi_valid = (
#                         phi_coeffs[valid_coeffs, 0]
#                         * energy_geom_mean ** phi_coeffs[valid_coeffs, 1]
#                     )
#
#                     theta_exceeds = fwhm_theta_valid > threshold
#                     phi_exceeds = fwhm_phi_valid > threshold
#                     either_exceeds = theta_exceeds | phi_exceeds
#
#                     # Get indices of valid events that exceed
#                     valid_event_indices = event_indices[valid_coeffs]
#                     scattering_quality_flags[valid_event_indices[either_exceeds]] |= (
#                         ImapDEScatteringUltraFlags.ABOVE_THRESHOLD.value
#                     )
#                 # Calculate FWHM using energy bin geometric mean
#                 fwhm_theta = theta_coeffs[:, 0] * energy_geom_mean ** theta_coeffs[:, 1]
#                 fwhm_phi = phi_coeffs[:, 0] * energy_geom_mean ** phi_coeffs[:, 1]
#                 # Check for NaN values
#
#                 # Don't flag NaN FWHM - treat NaN as if it passes
#                 has_nan_fwhm = np.isnan(fwhm_theta) | np.isnan(fwhm_phi)
#
#                 # Only check threshold for non-NaN events
#                 theta_exceeds = np.zeros(len(fwhm_theta), dtype=bool)
#                 phi_exceeds = np.zeros(len(fwhm_phi), dtype=bool)
#
#                 valid = ~has_nan_fwhm
#                 theta_exceeds[valid] = fwhm_theta[valid] > threshold
#                 phi_exceeds[valid] = fwhm_phi[valid] > threshold
#
#                 either_exceeds = theta_exceeds | phi_exceeds
#                 # has_nan = np.isnan(fwhm_theta) | np.isnan(fwhm_phi)
#                 # either_exceeds = either_exceeds | has_nan
#
#                 event_indices = np.where(event_mask)[0]
#                 scattering_quality_flags[event_indices[either_exceeds]] |= (
#                     ImapDEScatteringUltraFlags.ABOVE_THRESHOLD.value
#                 )
#
#                 # Debug output
#                 n_flagged = np.sum(either_exceeds)
#
#                 # Debug output for bin 0
#                 if ebin == 0:
#                     n_events = np.sum(event_mask)
#                     n_flagged = np.sum(either_exceeds)
#                     has_nan = np.isnan(fwhm_theta) | np.isnan(fwhm_phi)
#
#                     # Check accidentals
#                     bin_0_accidentals = ae_flux["accidental"].values[event_mask]
#                     n_accidentals = np.sum(bin_0_accidentals)
#
#                     if ebin == 0:
#                         has_nan = np.isnan(fwhm_theta) | np.isnan(fwhm_phi)
#
#                         # For NaN events, check their theta/phi values
#                         if np.any(has_nan):
#                             nan_indices = np.where(has_nan)[0]
#                             print("\n  NaN FWHM Debug:")
#                             print(f"  Total NaN events: {np.sum(has_nan)}")
#
#                             # Get original indices in full array
#                             original_indices = np.where(event_mask)[0][nan_indices]
#                             theta_nan = theta[original_indices]
#                             phi_nan = phi[original_indices]
#
#                             print(
#                                 f"  Theta for NaN events: min={theta_nan.min():.4f}, max={theta_nan.max():.4f}"
#                             )
#                             print(
#                                 f"  Phi for NaN events: min={phi_nan.min():.4f}, max={phi_nan.max():.4f}"
#                             )
#
#                             # Check coefficients
#                             print("  First 5 NaN coefficients:")
#                             for i in range(min(5, len(nan_indices))):
#                                 idx = nan_indices[i]
#                                 print(
#                                     f"    theta={theta[original_indices[i]]:.2f}, phi={phi[original_indices[i]]:.2f}, "
#                                     f"a_theta={theta_coeffs[idx, 0]:.4f}, g_theta={theta_coeffs[idx, 1]:.4f}"
#                                 )
#                                 # Check bin 0 specifically
#                                 bin_0_mask = (bin_indices == 0) & energy_in_range
#                                 bin_0_flagged = bin_0_mask & (
#                                     (scattering_quality_flags > 0)
#                                     | (outlier_quality_flags > 0)
#                                 )
#                                 print(
#                                     f"  Energy bin 0 (in range): {np.sum(bin_0_mask)} events, {np.sum(bin_0_flagged)} flagged, {np.sum(bin_0_mask) - np.sum(bin_0_flagged)} passing"
#                                 )
#                                 print("  ===================================\n")
#
#             # Get scatter values (for reference only)
#             scatter_values = ae_flux["scatter"].values.copy()
#
#             # Set ebin
#             # ebin = np.where(
#             #     (ae_flux["accidental"].values & (ae_flux["energy_sc"] > energy_max)),
#             #     255,
#             #     1,
#             # )
#             ebin = np.ones_like(ae_flux["accidental"].values.astype(np.uint16))
#
#             # Now create dataset with ALL original data (no filtering)
#             l1b_ds = xr.Dataset(
#                 {
#                     "event_times": (
#                         ["epoch"],
#                         ae_flux["tdb (s)"].values.astype(np.float32),
#                     ),
#                     "velocity_sc": (
#                         ["epoch", "component"],
#                         np.stack(
#                             [
#                                 ae_flux["v_x_sc"].values,
#                                 ae_flux["v_y_sc"].values,
#                                 ae_flux["v_z_sc"].values,
#                             ],
#                             axis=1,
#                         ).astype(np.float32),
#                     ),
#                     "velocity_dps_sc": (
#                         ["epoch", "component"],
#                         np.stack(
#                             [
#                                 ae_flux["v_x_sc"].values,
#                                 ae_flux["v_y_sc"].values,
#                                 ae_flux["v_z_sc"].values,
#                             ],
#                             axis=1,
#                         ).astype(np.float32),
#                     ),
#                     "velocity_dps_helio": (
#                         ["epoch", "component"],
#                         np.stack(
#                             [
#                                 ae_flux["v_x_hel"].values,
#                                 ae_flux["v_y_hel"].values,
#                                 ae_flux["v_z_hel"].values,
#                             ],
#                             axis=1,
#                         ).astype(np.float32),
#                     ),
#                     "energy_spacecraft": (
#                         ["epoch"],
#                         ae_flux["energy_sc"].values.astype(np.float32),
#                     ),
#                     "energy_heliosphere": (
#                         ["epoch"],
#                         ae_flux["energy_hel"].values.astype(np.float32),
#                     ),
#                     "energy": (
#                         ["epoch"],
#                         ae_flux["energy_sc"].values.astype(np.float32),
#                     ),
#                     "tof_energy": (
#                         ["epoch"],
#                         ae_flux["energy_sc"].values.astype(np.float32),
#                     ),
#                     "event_efficiency": (
#                         ["epoch"],
#                         ae_flux["eff"].values.astype(np.float64),
#                     ),
#                     "geometric_factor_blades": (
#                         ["epoch"],
#                         ae_flux["gf"].values.astype(np.float64),
#                     ),
#                     "quality_scattering": (
#                         ["epoch"],
#                         scattering_quality_flags,
#                     ),
#                     "quality_outliers": (
#                         ["epoch"],
#                         outlier_quality_flags,
#                     ),
#                     "scatter": (
#                         ["epoch"],
#                         scatter_values.astype(np.float32),
#                     ),
#                     "ebin": (
#                         ["epoch"],
#                         ebin,
#                     ),
#                     "theta": (
#                         ["epoch"],
#                         theta.astype(np.float32),
#                     ),
#                     "phi": (
#                         ["epoch"],
#                         phi.astype(np.float32),
#                     ),
#                 },
#                 coords={
#                     "epoch": ae_flux["tdb (s)"].values.astype(np.float64),
#                     "component": ["x", "y", "z"],
#                 },
#                 attrs=cdf_manager.get_global_attributes(
#                     f"imap_ultra_l1b_{id}sensor-de"
#                 ),
#             )
#
#             rates_ds = xr.Dataset(
#                 {
#                     "spin_phase": (
#                         ["epoch"],
#                         rates_flux["Spin Phase (deg)"].values.astype(np.float32),
#                     ),
#                     "start_rate": (
#                         ["epoch"],
#                         rates_flux["Start Rate (Hz)"].values.astype(np.float32),
#                     ),
#                     "stop_rate": (
#                         ["epoch"],
#                         rates_flux["Stop Rate (Hz)"].values.astype(np.float32),
#                     ),
#                     "coin_rate": (
#                         ["epoch"],
#                         rates_flux["Coin Rate (Hz)"].values.astype(np.float32),
#                     ),
#                     "dead_time_ratio": (
#                         ["epoch"],
#                         rates_flux["Dead Time Ratio"].values.astype(np.float32),
#                     ),
#                 },
#                 coords={
#                     "epoch": np.arange(len(rates_flux["Dead Time Ratio"].values)),
#                 },
#                 attrs=cdf_manager.get_global_attributes(
#                     f"imap_ultra_l1a_{id}sensor-rates"
#                 ),
#             )
#
#             print("Datasets created")
#             l1b_ds.attrs["Data_version"] = "000"
#             rates_ds.attrs["Data_version"] = "000"
#             l1b_ds.attrs["Repointing"] = f"repoint{int(pointing.replace('p', '')):05d}"
#             rates_ds.attrs["Repointing"] = (
#                 f"repoint{int(pointing.replace('p', '')):05d}"
#             )
#
#             write_cdf(l1b_ds)
#             write_cdf(rates_ds)
#
#             print(f"CDFs written for {pointing} pointing, sensor {id}\n")

Creating direct event dataset for p0 pointing and 45 sensor
  Processing 17299 events
  Calculating theta/phi from instrument direction vectors (Java method)...
  Theta range: [-52.42, 51.24] deg
  Phi range: [-59.97, 59.74] deg

  === APPLYING SCATTERING FILTER ===
[(3.0, 4.6), (4.6, 6.96), (6.96, 10.27), (10.27, 15.71), (15.71, 23.56), (23.56, 35.35), (35.35, 53.02), (53.02, 79.53), (79.53, 119.3), (119.3, 178.95), (178.95, 268.42)]
Energy range: 3.0 - 268.4 keV
Events in range: 15954/17299
Outlier quality flags: [1 1 0 ... 0 0 0]
Bin indices: [ 0 10  6 ...  0  2  1]

[LOOKUP BOUNDS]
Lookup table phi range: [-60.00, 60.00]
Lookup table theta range: [-60.00, 60.00]
Input phi range: [-59.97, 59.74]
Input theta range: [-52.42, 51.24]

  NaN FWHM Debug:
  Total NaN events: 330
  Theta for NaN events: min=-52.4164, max=51.2402
  Phi for NaN events: min=-59.9691, max=59.7376
  First 5 NaN coefficients:
    theta=15.02, phi=59.34, a_theta=nan, g_theta=nan
  Energy bin 0 (in range): 7929 eve

/Users/luco3133/projects/imap_processing/data/imap/ultra/l1b/2000/01/imap_ultra_l1b_45sensor-de_20000101-repoint00000_v000.cdf already exists, cannot create CDF file.  Returning...
/Users/luco3133/projects/imap_processing/data/imap/ultra/l1a/2000/01/imap_ultra_l1a_45sensor-rates_20000101-repoint00000_v000.cdf already exists, cannot create CDF file.  Returning...



[LOOKUP BOUNDS]
Lookup table phi range: [-60.00, 60.00]
Lookup table theta range: [-60.00, 60.00]
Input phi range: [-51.96, 44.80]
Input theta range: [-44.84, 34.43]

[LOOKUP BOUNDS]
Lookup table phi range: [-60.00, 60.00]
Lookup table theta range: [-60.00, 60.00]
Input phi range: [-54.62, 52.02]
Input theta range: [-39.16, 38.83]

[LOOKUP BOUNDS]
Lookup table phi range: [-60.00, 60.00]
Lookup table theta range: [-60.00, 60.00]
Input phi range: [-33.31, 44.81]
Input theta range: [-37.57, 39.24]

[LOOKUP BOUNDS]
Lookup table phi range: [-60.00, 60.00]
Lookup table theta range: [-60.00, 60.00]
Input phi range: [-28.41, 48.98]
Input theta range: [-20.10, 24.44]

[LOOKUP BOUNDS]
Lookup table phi range: [-60.00, 60.00]
Lookup table theta range: [-60.00, 60.00]
Input phi range: [-21.81, 19.80]
Input theta range: [-24.05, 24.94]
Datasets created
CDFs written for p0 pointing, sensor 45



In [ ]:
# ruff: noqa: E501
ae_flux = pd.read_csv(
    "/Users/luco3133/projects/ultra_stuff/validation_stuff"
    "/other_var_validation_20251024/ultra-45-inputs/AE-IMAP_ULTRA_45-p0.csv"
)
print(ae_flux.columns.tolist())

In [ ]:
# ruff: noqa: E501
# Check energy bin 1
species_dataset = pd.read_csv(
    "/Users/luco3133/projects/ultra_stuff/validation_stuff/other_var_validation_20251024/ultra-45-inputs/AE-IMAP_ULTRA_45-p0.csv"
)
mask = (species_dataset["energy_sc"] >= 0) & (species_dataset["energy_sc"] < 3.7)
print(f"Events in energy bin 1: {np.sum(mask)}")
print(f"Scatter mean: {species_dataset['scatter'][mask].mean():.2f} deg")
print(
    f"Scatter min/max: {species_dataset['scatter'][mask].min():.2f} / {species_dataset['scatter'][mask].max():.2f} deg"
)
# print(f"Quality flags (0=good): {np.sum(species_dataset['quality_scattering'][mask] == 0)} good, {np.sum(species_dataset['quality_scattering'][mask] != 0)} bad")

In [31]:
# ruff: noqa: E501
from pathlib import Path

search_path = Path("/Users/luco3133/projects/imap_processing/data/imap/ultra/l1c")
all_cdf_files = search_path.glob("**/*45sensor-space*v001.cdf")
print(sorted([path.name for path in all_cdf_files]))
# search_path = Path("/Users/luco3133/projects/imap_processing/data/imap/ultra/l1c")
# all_cdf_files = search_path.glob('**/*90sensor-space*v001.cdf')
# print(sorted([path.name for path in all_cdf_files]))

['imap_ultra_l1c_45sensor-spacecraftpset_20250416-repoint00000_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250417-repoint00001_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250418-repoint00002_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250419-repoint00003_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250420-repoint00004_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250421-repoint00005_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250422-repoint00006_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250423-repoint00007_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250424-repoint00008_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250425-repoint00009_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250426-repoint00010_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250427-repoint00011_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250428-repoint00012_v001.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250429-r

In [47]:
# ADD MISSING ATTRS
# cdf_attrs = ImapCdfAttributes()
# cdf_attrs.add_instrument_global_attrs("ultra")
# cdf_attrs.add_instrument_variable_attrs("ultra", "l1c")
#
# from pathlib import Path
#
# search_path = Path("/Users/luco3133/projects/imap_processing/data/imap/ultra/l1c")
# all_cdf_files = search_path.glob("**/*sensor-space*v001.cdf")
# print(all_cdf_files)
# cdfs = sorted([path for path in all_cdf_files])
#
#
# for file in cdfs:
#     cdf = load_cdf(file)
#
#     for var in cdf.variables:
#         try:
#             cdf[var].attrs = cdf_attrs.get_variable_attributes(var, check_schema=False)
#             print(f"updated: {var}")
#             print(cdf[var].attrs)
#         except:
#             print(var, "coultns do it")
#
#     cdf["epoch"].attrs = cdf_attrs.get_variable_attributes("epoch", check_schema=False)
#
#     cdf.attrs["Data_version"] = "002"
#     write_cdf(cdf)

<generator object Path.glob at 0x175e35690>
updated: counts
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': -9223372036854775808, 'FORMAT': 'I12', 'VALIDMIN': -9223372036854775808, 'VALIDMAX': 9223372036854775807, 'VAR_TYPE': 'data', 'UNITS': 'counts', 'CATDESC': 'Counts for a spin.', 'DEPEND_1': 'energy_bin_geometric_mean', 'DEPEND_2': 'pixel_index', 'FIELDNAM': 'counts', 'LABLAXIS': 'counts'}
updated: latitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'Latitude bin center corresponding to healpix index with range [-90, 90].', 'DEPEND_1': 'pixel_index', 'FIELDNAM': 'latitude', 'LABLAXIS': 'latitude'}
updated: longitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 

scatter_threshold
energy_delta_minus
energy_delta_plus
epoch
pixel_index
energy_bin_geometric_mean
spin_phase_step
updated: counts
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': -9223372036854775808, 'FORMAT': 'I12', 'VALIDMIN': -9223372036854775808, 'VALIDMAX': 9223372036854775807, 'VAR_TYPE': 'data', 'UNITS': 'counts', 'CATDESC': 'Counts for a spin.', 'DEPEND_1': 'energy_bin_geometric_mean', 'DEPEND_2': 'pixel_index', 'FIELDNAM': 'counts', 'LABLAXIS': 'counts'}
updated: latitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'Latitude bin center corresponding to healpix index with range [-90, 90].', 'DEPEND_1': 'pixel_index', 'FIELDNAM': 'latitude', 'LABLAXIS': 'latitude'}
updated: longitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -

energy_bin_delta
quality_flags
sensitivity
efficiency
geometric_function
dead_time_ratio
scatter_theta
scatter_phi
scatter_threshold
energy_delta_minus
energy_delta_plus
epoch
pixel_index
energy_bin_geometric_mean
spin_phase_step
updated: counts
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': -9223372036854775808, 'FORMAT': 'I12', 'VALIDMIN': -9223372036854775808, 'VALIDMAX': 9223372036854775807, 'VAR_TYPE': 'data', 'UNITS': 'counts', 'CATDESC': 'Counts for a spin.', 'DEPEND_1': 'energy_bin_geometric_mean', 'DEPEND_2': 'pixel_index', 'FIELDNAM': 'counts', 'LABLAXIS': 'counts'}
updated: latitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'Latitude bin center corresponding to healpix index with range [-90, 90].', 'DEPEND_1': 'pixel_index', 'FIELDNAM': 'latitude', 'LABLAXIS': 'latitude'}
updated: lo

energy_bin_delta
quality_flags
sensitivity
efficiency
geometric_function
dead_time_ratio
scatter_theta
scatter_phi
scatter_threshold
energy_delta_minus
energy_delta_plus
epoch
pixel_index
energy_bin_geometric_mean
spin_phase_step
updated: counts
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': -9223372036854775808, 'FORMAT': 'I12', 'VALIDMIN': -9223372036854775808, 'VALIDMAX': 9223372036854775807, 'VAR_TYPE': 'data', 'UNITS': 'counts', 'CATDESC': 'Counts for a spin.', 'DEPEND_1': 'energy_bin_geometric_mean', 'DEPEND_2': 'pixel_index', 'FIELDNAM': 'counts', 'LABLAXIS': 'counts'}
updated: latitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'Latitude bin center corresponding to healpix index with range [-90, 90].', 'DEPEND_1': 'pixel_index', 'FIELDNAM': 'latitude', 'LABLAXIS': 'latitude'}
updated: lo

energy_bin_delta
quality_flags
sensitivity
efficiency
geometric_function
dead_time_ratio
scatter_theta
scatter_phi
scatter_threshold
energy_delta_minus
energy_delta_plus
epoch
pixel_index
energy_bin_geometric_mean
spin_phase_step
updated: counts
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': -9223372036854775808, 'FORMAT': 'I12', 'VALIDMIN': -9223372036854775808, 'VALIDMAX': 9223372036854775807, 'VAR_TYPE': 'data', 'UNITS': 'counts', 'CATDESC': 'Counts for a spin.', 'DEPEND_1': 'energy_bin_geometric_mean', 'DEPEND_2': 'pixel_index', 'FIELDNAM': 'counts', 'LABLAXIS': 'counts'}
updated: latitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'Latitude bin center corresponding to healpix index with range [-90, 90].', 'DEPEND_1': 'pixel_index', 'FIELDNAM': 'latitude', 'LABLAXIS': 'latitude'}
updated: lo

...

updated: counts
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': -9223372036854775808, 'FORMAT': 'I12', 'VALIDMIN': -9223372036854775808, 'VALIDMAX': 9223372036854775807, 'VAR_TYPE': 'data', 'UNITS': 'counts', 'CATDESC': 'Counts for a spin.', 'DEPEND_1': 'energy_bin_geometric_mean', 'DEPEND_2': 'pixel_index', 'FIELDNAM': 'counts', 'LABLAXIS': 'counts'}
updated: latitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'Latitude bin center corresponding to healpix index with range [-90, 90].', 'DEPEND_1': 'pixel_index', 'FIELDNAM': 'latitude', 'LABLAXIS': 'latitude'}
updated: longitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'L

updated: counts
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': -9223372036854775808, 'FORMAT': 'I12', 'VALIDMIN': -9223372036854775808, 'VALIDMAX': 9223372036854775807, 'VAR_TYPE': 'data', 'UNITS': 'counts', 'CATDESC': 'Counts for a spin.', 'DEPEND_1': 'energy_bin_geometric_mean', 'DEPEND_2': 'pixel_index', 'FIELDNAM': 'counts', 'LABLAXIS': 'counts'}
updated: latitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'Latitude bin center corresponding to healpix index with range [-90, 90].', 'DEPEND_1': 'pixel_index', 'FIELDNAM': 'latitude', 'LABLAXIS': 'latitude'}
updated: longitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'L

updated: counts
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': -9223372036854775808, 'FORMAT': 'I12', 'VALIDMIN': -9223372036854775808, 'VALIDMAX': 9223372036854775807, 'VAR_TYPE': 'data', 'UNITS': 'counts', 'CATDESC': 'Counts for a spin.', 'DEPEND_1': 'energy_bin_geometric_mean', 'DEPEND_2': 'pixel_index', 'FIELDNAM': 'counts', 'LABLAXIS': 'counts'}
updated: latitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'Latitude bin center corresponding to healpix index with range [-90, 90].', 'DEPEND_1': 'pixel_index', 'FIELDNAM': 'latitude', 'LABLAXIS': 'latitude'}
updated: longitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'L

updated: counts
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': -9223372036854775808, 'FORMAT': 'I12', 'VALIDMIN': -9223372036854775808, 'VALIDMAX': 9223372036854775807, 'VAR_TYPE': 'data', 'UNITS': 'counts', 'CATDESC': 'Counts for a spin.', 'DEPEND_1': 'energy_bin_geometric_mean', 'DEPEND_2': 'pixel_index', 'FIELDNAM': 'counts', 'LABLAXIS': 'counts'}
updated: latitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'Latitude bin center corresponding to healpix index with range [-90, 90].', 'DEPEND_1': 'pixel_index', 'FIELDNAM': 'latitude', 'LABLAXIS': 'latitude'}
updated: longitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'L

updated: counts
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': -9223372036854775808, 'FORMAT': 'I12', 'VALIDMIN': -9223372036854775808, 'VALIDMAX': 9223372036854775807, 'VAR_TYPE': 'data', 'UNITS': 'counts', 'CATDESC': 'Counts for a spin.', 'DEPEND_1': 'energy_bin_geometric_mean', 'DEPEND_2': 'pixel_index', 'FIELDNAM': 'counts', 'LABLAXIS': 'counts'}
updated: latitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'Latitude bin center corresponding to healpix index with range [-90, 90].', 'DEPEND_1': 'pixel_index', 'FIELDNAM': 'latitude', 'LABLAXIS': 'latitude'}
updated: longitude
{'DEPEND_0': 'epoch', 'DISPLAY_TYPE': 'time_series', 'FILLVAL': '-1.0e31', 'FORMAT': 'F12.6', 'VALIDMIN': -3.4028235e+38, 'VALIDMAX': 3.4028235e+38, 'VAR_TYPE': 'data', 'UNITS': 'degrees', 'dtype': 'float32', 'CATDESC': 'L

In [46]:
search_path = Path("/Users/luco3133/projects/imap_processing/data/imap/ultra/l1c")
all_cdf_files = search_path.glob("**/*sensor-space*v002.cdf")

for cdf_file in all_cdf_files:
    print(f"Deleting {cdf_file}")
    cdf_file.unlink()

Deleting /Users/luco3133/projects/imap_processing/data/imap/ultra/l1c/2025/04/imap_ultra_l1c_45sensor-spacecraftpset_20250426-repoint00010_v002.cdf
Deleting /Users/luco3133/projects/imap_processing/data/imap/ultra/l1c/2025/04/imap_ultra_l1c_90sensor-spacecraftpset_20250419-repoint00003_v002.cdf
Deleting /Users/luco3133/projects/imap_processing/data/imap/ultra/l1c/2025/04/imap_ultra_l1c_90sensor-spacecraftpset_20250426-repoint00010_v002.cdf
Deleting /Users/luco3133/projects/imap_processing/data/imap/ultra/l1c/2025/04/imap_ultra_l1c_45sensor-spacecraftpset_20250419-repoint00003_v002.cdf
Deleting /Users/luco3133/projects/imap_processing/data/imap/ultra/l1c/2025/04/imap_ultra_l1c_45sensor-spacecraftpset_20250429-repoint00013_v002.cdf
Deleting /Users/luco3133/projects/imap_processing/data/imap/ultra/l1c/2025/04/imap_ultra_l1c_90sensor-spacecraftpset_20250421-repoint00005_v002.cdf
Deleting /Users/luco3133/projects/imap_processing/data/imap/ultra/l1c/2025/04/imap_ultra_l1c_90sensor-spacecraft